In [1]:
from datasets import load_from_disk
from tqdm import tqdm

/home/kurogane/miniforge3/envs/unslothbw/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dir_cache = r"/media/kurogane/HD-NRLD-A/cache"

In [3]:
ds_reconst = load_from_disk("OpenScience-OS-Q3-235B-4")

In [4]:
model_id = "sbintuitions/sarashina2.2-3b-instruct-v0.1"
chat_template = "sarashina22"

In [5]:
from transformers import AutoTokenizer

tokenizer_transformers = AutoTokenizer.from_pretrained(
    model_id,
    cache_dir=dir_cache
    )


# Unsloth

In [6]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 3600 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",      # Llama-3.1 2x faster
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-405B-bnb-4bit",    # 4bit for 405b!
    "unsloth/Mistral-Small-Instruct-2409",     # Mistral 22b 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!

    "unsloth/Llama-3.2-1B-bnb-4bit",           # NEW! Llama 3.2 models
    "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
    "unsloth/Llama-3.2-3B-bnb-4bit",
    "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",

    "unsloth/Llama-3.3-70B-Instruct-bnb-4bit" # NEW! Llama 3.3 70B!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_id, # or choose "unsloth/Llama-3.2-1B-Instruct"
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    cache_dir=dir_cache,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

/tmp/ipykernel_138471/2133455727.py:1: UserWarning: WARNING: Unsloth should be imported before transformers to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 07-20 23:27:13 [__init__.py:244] Automatically detected platform cuda.
==((====))==  Unsloth 2025.7.3: Fast Llama patching. Transformers: 4.52.4. vLLM: 0.9.2.
   \\   /|    NVIDIA GeForce RTX 5090. Num GPUs = 1. Max memory: 31.324 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.3.1
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards: 100%|██████████| 2/2 [00:03<00:00,  1.68s/it]


In [7]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2025.7.3 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [8]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = chat_template,
)

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }
pass



In [9]:
from unsloth.chat_templates import standardize_sharegpt
dataset = standardize_sharegpt(ds_reconst)
dataset = dataset.map(formatting_prompts_func, batched = True,)

In [10]:
dataset[5]["conversations"]

[{'content': 'A researcher is analyzing the gut microbiome of a patient using shotgun metagenomic sequencing but notices an unusually high proportion of eukaryotic viral sequences. Subsequent validation reveals that the sequences do not originate from the host genome or known human pathogens. Which of the following factors is most likely to explain this observation and would require a multi-step correction strategy?  \nA: Contamination from DNA extraction kit reagents containing trace eukaryotic viral DNA  \nB: Degradation of microbial DNA due to improper sample storage, leaving only viral DNA intact  \nC: Cross-species assembly errors during bioinformatic processing, misclassifying host-derived sequences as viral  \nD: Presence of symbiotic eukaryotic parasites producing extracellular vesicles encapsulating viral DNA',
  'role': 'user'},
 {'content': 'The observation of an unusually high proportion of eukaryotic viral sequences in shotgun metagenomic sequencing, not originating from t

In [11]:
dataset[5]["text"]

'<|user|>A researcher is analyzing the gut microbiome of a patient using shotgun metagenomic sequencing but notices an unusually high proportion of eukaryotic viral sequences. Subsequent validation reveals that the sequences do not originate from the host genome or known human pathogens. Which of the following factors is most likely to explain this observation and would require a multi-step correction strategy?  \nA: Contamination from DNA extraction kit reagents containing trace eukaryotic viral DNA  \nB: Degradation of microbial DNA due to improper sample storage, leaving only viral DNA intact  \nC: Cross-species assembly errors during bioinformatic processing, misclassifying host-derived sequences as viral  \nD: Presence of symbiotic eukaryotic parasites producing extracellular vesicles encapsulating viral DNA</s><|assistant|>The observation of an unusually high proportion of eukaryotic viral sequences in shotgun metagenomic sequencing, not originating from the host or known pathoge

In [12]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported


EPOCHS    = 2                          # 1〜3 が安全
BATCH     = 8                          # per‑device
GRAD_ACC  = 16                          # 2×8=16 → EBS
TOTAL_STEPS = int(2565 * EPOCHS)       # 上記計算値
LEARNING_RATE = 2e-4

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = BATCH,
        gradient_accumulation_steps = GRAD_ACC,
        warmup_steps = int(TOTAL_STEPS * 0.05),
        num_train_epochs = EPOCHS, # Set this for 1 full training run.
        # max_steps = 60,
        learning_rate = LEARNING_RATE,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10000,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc
    ),
)

In [13]:
chat = [
  {"role": "user", "content": "Hello, how are you?"},
  {"role": "assistant", "content": "I'm doing great. How can I help you today?"},
  {"role": "user", "content": "I'd like to show off how chat templating works!"},
]

print(tokenizer_transformers.apply_chat_template(chat, tokenize=False))

<|user|>Hello, how are you?</s><|assistant|>I'm doing great. How can I help you today?</s><|user|>I'd like to show off how chat templating works!</s>


In [14]:
tokenizer_transformers.chat_template

'\n{%- set user_messages = messages | selectattr(\'role\', \'equalto\', \'user\') | list %}\n{%- macro output_available_tools(tools, message) %}\n{%- if tools and (message == user_messages[-1]) %}\n    {{- \'<|available_tools|>[\' }}\n    {%- for tool in tools %}\n        {%- set tool = tool.function %}\n        {{- "{" }}\n        {%- for key, val in tool.items() if key != "return" %}\n            {%- if val is string %}\n                {{- "\'" + key + "\': \'" + val + "\'" }}\n            {%- else %}\n                {{- "\'" + key + "\': " + val|string }}\n            {%- endif %}\n            {%- if not loop.last %}\n                {{- ", " }}\n            {%- endif %}\n        {%- endfor %}\n        {{- "}" }}\n        {%- if not loop.last %}\n            {{- ", " }}\n        {%- else %}\n            {{- "]" }}\n        {%- endif %}\n    {%- endfor %}\n    {{- eos_token -}}\n{%- endif %}\n{%- endmacro %}\n\n{%- macro output_tool_results(tool_results) %}\n{{- \'<|tool_results|>[

In [15]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|user|>",
    response_part = "<|assistant|>",
)

In [16]:
tokenizer.decode(trainer.train_dataset[5]["input_ids"])

'<|user|>A researcher is analyzing the gut microbiome of a patient using shotgun metagenomic sequencing but notices an unusually high proportion of eukaryotic viral sequences. Subsequent validation reveals that the sequences do not originate from the host genome or known human pathogens. Which of the following factors is most likely to explain this observation and would require a multi-step correction strategy?  \nA: Contamination from DNA extraction kit reagents containing trace eukaryotic viral DNA  \nB: Degradation of microbial DNA due to improper sample storage, leaving only viral DNA intact  \nC: Cross-species assembly errors during bioinformatic processing, misclassifying host-derived sequences as viral  \nD: Presence of symbiotic eukaryotic parasites producing extracellular vesicles encapsulating viral DNA</s><|assistant|>The observation of an unusually high proportion of eukaryotic viral sequences in shotgun metagenomic sequencing, not originating from the host or known pathoge

In [17]:
space = tokenizer(" ", add_special_tokens = False).input_ids[0]
tokenizer.decode([space if x == -100 else x for x in trainer.train_dataset[5]["labels"]])

'                                                                                                                                                                                                         The observation of an unusually high proportion of eukaryotic viral sequences in shotgun metagenomic sequencing, not originating from the host or known pathogens, is most likely explained by **contamination from DNA extraction kit reagents containing trace eukaryotic viral DNA** (Option A). This is a well-documented issue in microbiome studies, where reagent contamination can introduce foreign DNA, leading to false positives. Addressing this requires a multi-step strategy: identifying the contaminant sources (e.g., using negative controls), switching to low-biomass or contaminant-free kits, and applying bioinformatic tools to filter out known contaminant sequences. \n\nOther options are less plausible:  \n- **Option B** (degradation of microbial DNA) is unlikely because improper storage 

In [18]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA GeForce RTX 5090. Max memory = 31.324 GB.
3.189 GB of memory reserved.


In [19]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 315,579 | Num Epochs = 2 | Total steps = 4,932
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 16 x 1) = 128
 "-____-"     Trainable parameters = 26,869,760 of 3,382,479,360 (0.79% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss


/home/kurogane/miniforge3/envs/unslothbw/lib/python3.12/site-packages/peft/utils/other.py:1221: UserWarning: Unable to fetch remote file due to the following error (ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 1dd0e47d-f373-4728-b137-e6b81317d563)') - silently ignoring the lookup for the file config.json in sbintuitions/sarashina2.2-3b-instruct-v0.1.
  warnings.warn(
/home/kurogane/miniforge3/envs/unslothbw/lib/python3.12/site-packages/peft/utils/save_and_load.py:238: UserWarning: Could not find a config file in sbintuitions/sarashina2.2-3b-instruct-v0.1 - will assume that the vocabulary was not modified.
  warnings.warn(
/home/kurogane/miniforge3/envs/unslothbw/lib/python3.12/site-packages/peft/utils/other.py:1221: UserWarning: Unable to fetch remote file due to the following error (ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 1879

In [20]:
dir_save_lora = "sci_300k/lora_model"
dir_save_model = "sci_300k/model"

In [21]:
model.save_pretrained(dir_save_lora)  # Local saving
tokenizer.save_pretrained(dir_save_lora)


('sci_300k/lora_model/tokenizer_config.json',
 'sci_300k/lora_model/special_tokens_map.json',
 'sci_300k/lora_model/chat_template.jinja',
 'sci_300k/lora_model/tokenizer.model',
 'sci_300k/lora_model/added_tokens.json',
 'sci_300k/lora_model/tokenizer.json')

In [23]:
model.save_pretrained_merged(dir_save_model, tokenizer, save_method = "merged_16bit",)

Found HuggingFace hub cache directory: /home/kurogane/.cache/huggingface/hub
Checking cache directory for required files...
Successfully copied all 2 files from cache to sci_300k/model.


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [00:11<00:00,  5.71s/it]
